# SRPS: Semantic-aware SAE Steering Demo

This notebook demonstrates SRPS (Semantic-Robust Prompt Steering) using the unified steering module.

SRPS uses semantic token masking to focus on meaningful tokens when computing SAE latent averages, combined with roleplay prompts to steer model behavior.

In [9]:
# SRPS: Semantic-aware SAE Steering
# Using the unified steering module

import os
os.environ.pop('CUDA_VISIBLE_DEVICES', None)
import numpy as np
import torch
import sys
from pathlib import Path
# make sure the repo root / Steering package is on the import path
sys.path.append(str(Path.cwd().parent.resolve()))
from Steering import SteeringPipeline, PipelineConfig

## 1. Initialize Pipeline

In [10]:
# Create pipeline with model configuration
config = PipelineConfig.load('/home/aiotlab/mnt/hoplt/Benchmark/Configs/Eval/srps_roleplay_gemma.json')
# Authenticate with HuggingFace
pipeline = SteeringPipeline(config)


/home/aiotlab/mnt/hoplt/Benchmark/Steering/config/pipeline.py:151: UserWarning: SteerConfig: ignoring unknown keys ['n_test']. Check for typos or stale config fields.
  steer_config = SteerConfig.from_dict(steer_data)


## 3. Extract Steering Vector

SRPS extracts steering vectors by:
1. Masking stopwords, punctuation, and BOS tokens (semantic filtering)
2. Computing mean SAE activations over semantic tokens only
3. Scoring features by: `score = (target_mean - contrast_mean) + β * (target_active - contrast_active)`
4. Selecting top-k features and decoding to residual space

In [3]:
import os
import torch

print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("Device count =", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

CUDA_VISIBLE_DEVICES = None
Device count = 3
0 NVIDIA A30
1 NVIDIA A30
2 NVIDIA A30


In [4]:
pipeline.setup()

2026-03-06 14:59:25 | Steering.pipeline                   | INFO     | Authenticating with HuggingFace...


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


2026-03-06 14:59:25 | Steering.pipeline                   | INFO     | Authentication successful
2026-03-06 14:59:25 | Steering.pipeline                   | INFO     | Loading model: google/gemma-2-2b (sae)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer
2026-03-06 14:59:59 | Steering.data.loader                | INFO     | Loaded 3000 composite samples from srps_roleplay_gms8k
2026-03-06 14:59:59 | Steering.data.loader                | INFO     | Loaded 3000 samples from srps_roleplay_gms8k
2026-03-06 14:59:59 | Steering.pipeline                   | INFO     | Attempting to load SAE from gemma-scope-2b-pt-res-canonical, id=layer_25/width_16k/canonical (on CPU)


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


2026-03-06 15:17:14 | Steering.pipeline                   | INFO     | Saving extracted vector to: Vector/SRPS/gms8k.pt
2026-03-06 15:17:14 | Steering.pipeline                   | INFO     | Extracted SRPS vector: {'method': 'SRPS', 'layer': [25], 'top_k': 15}
2026-03-06 15:17:14 | Steering.pipeline                   | INFO     | Initialized SRPS steer model at layer [25]


## 4. Visualize Top SAE Features

Inspect the most important features that distinguish roleplay from baseline.

In [11]:
from IPython.display import IFrame

html_template = "https://neuronpedia.org/{}/{}/{}?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"

def show_neuronpedia(feature_idx: int, layer: int = config.extractor.layer[0]):
    """Display a feature on Neuronpedia."""
    html = html_template.format(
        "gemma-2-2b",
        f"{layer}-gemmascope-res-16k",
        feature_idx
    )
    return IFrame(html, width=1200, height=300)

# Display top features with their scores
print("Top discriminative SAE features for roleplay steering:")
for idx in pipeline.extractor.top_idx:
    score = pipeline.extractor.score[idx].item()
    print(f"\nFeature {idx}, score = {score:.3f}")
    display(show_neuronpedia(int(idx)))

Top discriminative SAE features for roleplay steering:


AttributeError: 'NoneType' object has no attribute 'top_idx'

## 5. Create Steered Model and Generate

The `SRPSSteerModel` applies steering while preserving the residual norm:

$$\text{resid}' = (\text{resid} + \lambda \cdot v) \cdot \frac{\|\text{resid}\|}{\|\text{resid} + \lambda \cdot v\|}$$

This prevents the steering from disrupting the model's internal scale.

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()
pipeline.run()

2026-03-06 15:31:12 | Steering.pipeline                   | INFO     | Authenticating with HuggingFace...


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


2026-03-06 15:31:12 | Steering.pipeline                   | INFO     | Authentication successful
2026-03-06 15:31:12 | Steering.pipeline                   | INFO     | Skipping extraction - loading vector from: Vector/SRPS/gms8k.pt
2026-03-06 15:31:12 | Steering.pipeline                   | INFO     | Loading model: google/gemma-2-2b (sae)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer


/home/aiotlab/mnt/hoplt/Benchmark/Steering/config/results.py:66: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path, map_location="cpu")  # load to CPU fir

2026-03-06 15:31:36 | Steering.pipeline                   | INFO     | Attempting to load SAE from gemma-scope-2b-pt-res-canonical, id=layer_25/width_16k/canonical (on CPU)
2026-03-06 15:31:39 | Steering.pipeline                   | INFO     | Initialized SRPS steer model at layer [25]
2026-03-06 15:31:39 | Steering.data.loader                | INFO     | Loaded 5 samples from gms8k


Generating: 100%|██████████| 5/5 [01:48<00:00, 21.76s/it]


{'results': [{'prompt': "Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?\nLet's think step by step:",
   'response': ["\n\n1. Janet eats 3 eggs per day.\n2. She bakes muffins for her friends with 4 eggs.\n3. She sells the remainder at the farmers' market for $2 per fresh duck egg.\n4. She makes $2 per egg.\n5. She makes $2 x 16 = $32 per day.\n\nTherefore, Janet makes $32 per day at the farmers' market.<eos>"],
   'sample': {'question': "Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?\nLet's think step by step:",
    'answer': "

: 

In [7]:
# Compare steered vs non-steered generation
prompt = TEST_PROMPTS[0]

print(f"Prompt: {prompt}\n")
print("=" * 60)

# Non-steered baseline
print("\nBaseline (no steering):")
print(pipeline.generate(prompt, coeff=0.0, max_new_tokens=50, apply_steer=False))

# Steered with roleplay persona
print("\nSteered (coeff=2.0) - should adopt roleplay persona:")
print(pipeline.generate(prompt, coeff=15.0, max_new_tokens=50))

NameError: name 'TEST_PROMPTS' is not defined

## 6. Coefficient Sweep

Explore how different steering strengths affect the roleplay behavior.

In [ ]:
# Coefficient sweep
prompt = TEST_PROMPTS[2]

print(f"Prompt: {prompt}\n")
print("=" * 60)

for coeff in np.arange(0, 21, 5):
    print(f"\nCoefficient = {coeff}")
    print("-" * 40)
    response = pipeline.generate(
        prompt, 
        coeff=coeff, 
        max_new_tokens=150,
        apply_steer=(coeff != 0),
    )
    print(response)

## 7. Batch Evaluation

Test steering across multiple prompts from the base dataset.

In [ ]:
from tqdm import tqdm

# Test on multiple prompts
test_prompts = contrast_data[:20]  # First 20 questions

results = {
    'baseline': [],
    'steered': [],
}

for prompt in tqdm(test_prompts[:10], desc="Generating"):  # First 10 for demo
    # Baseline
    baseline = pipeline.generate(prompt, coeff=0.0, max_new_tokens=50, apply_steer=False)
    results['baseline'].append({'input': prompt, 'output': baseline})
    
    # Steered
    steered = pipeline.generate(prompt, coeff=2.0, max_new_tokens=50)
    results['steered'].append({'input': prompt, 'output': steered})

# Show sample results
print("=" * 60)
print("Sample Comparisons:")
print("=" * 60)
for i in range(min(3, len(results['baseline']))):
    print(f"\nPrompt: {results['baseline'][i]['input'][:80]}...")
    print(f"\nBaseline: {results['baseline'][i]['output'][:120]}...")
    print(f"\nSteered:  {results['steered'][i]['output'][:120]}...")
    print("-" * 40)

## 8. Try Different Roleplay Types

Switch between arithmetic and commonsense roleplay.

In [ ]:
# Try commonsense roleplay (CSQA)
# Uncomment to run

# csqa_roleplay = loader.load("roleplay", "csqa")
# csqa_data = loader.load("qa", "csqa")
# 
# csqa_contrast = [data['question'] for data in csqa_data][:300]
# csqa_target = [
#     np.random.choice(csqa_roleplay) + "\n" + data['question']
#     for data in csqa_data
# ][:300]
# 
# # Create new extractor for CSQA
# csqa_extractor = SRPSExtractor(
#     model=pipeline.model,
#     sae=sae,
#     layer=TARGET_LAYER,
#     batch_size=8,
#     act_threshold=0.2,
#     top_k=15,
# )
# 
# csqa_vector = csqa_extractor.extract(csqa_target, csqa_contrast, beta=5.0)
# print(f"CSQA steering vector extracted with {len(csqa_extractor.top_idx)} features")

## 9. Save Results

In [ ]:
# Save steering vector and results
output_dir = f"srps_results_{ROLEPLAY_TYPE}_layer{TARGET_LAYER}"
os.makedirs(output_dir, exist_ok=True)

torch.save({
    'steering_vector': steering_vector.cpu(),
    'sparse_latent': extractor.sparse_latent.cpu(),
    'top_idx': extractor.top_idx.cpu(),
    'score': extractor.score.cpu(),
    'metadata': extractor.metadata,
    'roleplay_type': ROLEPLAY_TYPE,
}, f"{output_dir}/srps_extractor_results.pt")

print(f"Results saved to {output_dir}/")